# 04.3 · Modelagem — regras de associação

**CRISP-DM fase 4 — modelagem** · Turismo e Hotelaria

---

Entrada: a base pré-processada do notebook [`03-preparacao`](03-preparacao.ipynb).

As regras de associação tratam cada reserva como uma **cesta de características**
(tipo de hotel, mês de chegada, canal, faixa de antecedência...). Elas procuram
combinações que aparecem juntas com frequência e as transformam em regras
**se A, então B**. É a pergunta 3 do PBL: *que padrões de coocorrência sustentam
ou contradizem a segmentação?*

| Algoritmo | Como encontra os conjuntos frequentes |
|---|---|
| **Apriori** | nível a nível: gera candidatos de tamanho *k* a partir dos frequentes de tamanho *k*−1 e conta cada um na base |
| **Eclat** | vertical: guarda, para cada item, a lista de reservas que o têm, e conta interseções dessas listas em profundidade |
| **FP-Growth** | comprime a base numa árvore de prefixos (FP-tree) e minera a árvore, sem gerar candidatos |

Os três **encontram exatamente os mesmos conjuntos**; o que muda é o tempo. Por
isso o notebook roda os três, confere que o resultado bate e só então gera as
regras.

As métricas de cada regra A → B:
* **suporte:** fração das reservas com A e B juntos;
* **confiança:** entre as reservas com A, a fração que também tem B;
* **lift:** quantas vezes A aumenta a chance de B. Lift 1 significa que A e B
  são independentes.

In [ ]:
import time
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from mlxtend.frequent_patterns import apriori, fpgrowth  # pacote mlxtend

warnings.filterwarnings("ignore")  # silencia avisos de depreciação do mlxtend

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MIN_SUPORTE = 0.15  # conjunto precisa aparecer em ao menos 15% das reservas
MAX_ITENS = 3  # tamanho máximo dos conjuntos (regras com até 3 itens)
SUPORTE_MAXIMO = 0.90  # itens em mais de 90% das reservas não informam nada
MIN_CONFIANCA = 0.60  # regra precisa acertar em ao menos 60% dos casos
MIN_LIFT = 1.20  # e aumentar em ao menos 20% a chance do consequente
TOP = 15  # quantas regras aparecem nas tabelas

## 1 · Dos dados às transações

Cada coluna da saída do `03` vira um ou mais **itens**:
* **binárias** (One-Hot e indicadores `room_changed`, `has_special_requests`,
  `is_repeated_guest`...): viram item quando valem 1, como `hotel=City Hotel`
  ou `has_special_requests`;
* **numéricas** (contínuas e discretas escalonadas): são cortadas em
  **tercis** (baixo, médio, alto). O MinMax do 03 preserva a ordem dos
  valores, então os tercis são os mesmos da base original.

`is_canceled` não entrou na base pré-processada — foi reservado na preparação,
para não ser usado como atributo de entrada do agrupamento. Aqui ele volta,
só como item, direto da base tratada: é o melhor candidato a consequente —
*características → cancela ou não cancela* — e isso não é vazamento, porque
nenhum cluster é formado com ele.

In [ ]:
pp_path = RAIZ / "data" / "processed" / "hotel_bookings_preprocessed.csv"
tr_path = RAIZ / "data" / "processed" / "hotel_bookings_tratado.csv"

base = pd.read_csv(pp_path, sep=";")  # saída do 03
df = pd.read_csv(tr_path, sep=";")  # base tratada: nomes originais e o rótulo reservado
assert len(base) == len(df), "base pré-processada e base tratada desalinhadas"

variavel = {
    c: max((o for o in df.columns if c == o or c.startswith(f"{o}_")), key=len, default=c)
    for c in base.columns
}  # cada feature → variável de origem (desfaz o one-hot)


def faixas(serie):
    # Corta uma coluna numérica em tercis; colunas quase constantes ficam com 2 faixas.
    bordas = np.unique(serie.quantile([0, 1 / 3, 2 / 3, 1]))
    if len(bordas) < 3:  # quase constante: não vira item
        return None
    rotulos = {2: ["baixo", "alto"], 3: ["baixo", "medio", "alto"]}[len(bordas) - 1]
    return pd.cut(serie, bordas, labels=rotulos, include_lowest=True)


itens = {}
for coluna in base.columns:
    valores = base[coluna]
    if set(valores.unique()) <= {0, 1}:  # binária: item quando vale 1
        categoria = coluna[len(variavel[coluna]) + 1 :]  # nome após a variável de origem
        itens[f"{variavel[coluna]}={categoria}" if categoria else coluna] = valores == 1
    elif (corte := faixas(valores)) is not None:  # numérica: um item por faixa
        for faixa in corte.cat.categories:
            itens[f"{coluna}={faixa}"] = corte == faixa

# O rótulo reservado volta só como item, não como atributo de entrada
itens["is_canceled=sim"] = df["is_canceled"].eq(1)
itens["is_canceled=nao"] = df["is_canceled"].eq(0)

transacoes = pd.DataFrame(itens)  # reservas × itens, True quando a reserva tem o item
suporte_item = transacoes.mean()  # fração de reservas com cada item
transacoes = transacoes.loc[:, suporte_item <= SUPORTE_MAXIMO]  # descarta itens quase universais
N = len(transacoes)

descartados = sorted(suporte_item[suporte_item > SUPORTE_MAXIMO].index)
print(
    f"{N:,} reservas · {len(itens)} itens gerados · {transacoes.shape[1]} mantidos "
    f"(descartados por estarem em mais de {SUPORTE_MAXIMO:.0%}: {descartados})"
)
print(f"Itens com suporte ≥ {MIN_SUPORTE:.0%}: {(transacoes.mean() >= MIN_SUPORTE).sum()}")

## 2 · Os três algoritmos

O `mlxtend` traz Apriori e FP-Growth, mas não o Eclat. A implementação abaixo
faz a busca em profundidade sobre as listas verticais, e a interseção de duas
listas é um `&` entre vetores booleanos.

In [ ]:
def eclat(transacoes, min_suporte, max_itens):
    vertical = {
        item: transacoes[item].to_numpy()
        for item in transacoes.columns
        if transacoes[item].mean() >= min_suporte
    }  # item → em quais reservas ele aparece
    encontrados = []

    def expandir(prefixo, presenca, candidatos):
        for i, item in enumerate(candidatos):
            juntos = presenca & vertical[item]  # reservas com o prefixo E o item
            suporte = juntos.mean()
            if suporte >= min_suporte:  # frequente: guarda e tenta crescer
                conjunto = prefixo + [item]
                encontrados.append((suporte, frozenset(conjunto)))
                if len(conjunto) < max_itens:
                    # só itens à frente: evita repetir conjuntos
                    expandir(conjunto, juntos, candidatos[i + 1 :])

    expandir([], np.ones(len(transacoes), dtype=bool), sorted(vertical))
    return pd.DataFrame(encontrados, columns=["support", "itemsets"])


def rodar_fpgrowth():
    return fpgrowth(transacoes, min_support=MIN_SUPORTE, max_len=MAX_ITENS, use_colnames=True)


algoritmos = {  # nome: função que devolve os conjuntos frequentes (support e itemsets)
    "Apriori": lambda: apriori(
        transacoes, min_support=MIN_SUPORTE, max_len=MAX_ITENS, use_colnames=True, low_memory=True
    ),
    "Eclat": lambda: eclat(transacoes, MIN_SUPORTE, MAX_ITENS),
    "FP-Growth": rodar_fpgrowth,
}

resultados, tempos = {}, {}
for nome, rodar in algoritmos.items():
    inicio = time.perf_counter()
    resultados[nome] = rodar()  # conjuntos frequentes
    tempos[nome] = time.perf_counter() - inicio
    print(f"{nome}: {len(resultados[nome]):,} conjuntos em {tempos[nome]:.2f}s")

In [ ]:
# Os três precisam encontrar os mesmos conjuntos, com o mesmo suporte
referencia = resultados["FP-Growth"].set_index("itemsets")["support"]
for nome, itemsets in resultados.items():
    atual = itemsets.set_index("itemsets")["support"]
    assert set(atual.index) == set(referencia.index), f"{nome} encontrou conjuntos diferentes"
    iguais = np.allclose(atual.loc[referencia.index], referencia)
    assert iguais, f"{nome} calculou suportes diferentes"
print("Os três algoritmos encontraram os mesmos conjuntos frequentes.")

# o que muda entre eles é o tempo
ax = pd.Series(tempos).sort_values().plot.barh(figsize=(7, 2.5), color="tab:blue")
titulo = f"Tempo para minerar {len(referencia):,} conjuntos".replace(",", ".")
ax.set(title=titulo, xlabel="segundos")
ax.grid(alpha=0.3, axis="x")
plt.show()

## 3 · Regras

As regras saem dos conjuntos frequentes, que são os mesmos para os três
algoritmos. As fórmulas ficam explícitas na função.

In [ ]:
def gerar_regras(itemsets):
    # suporte de cada conjunto frequente
    suporte = dict(zip(itemsets["itemsets"], itemsets["support"]))
    regras = []
    for conjunto, sup in suporte.items():
        for tamanho in range(1, len(conjunto)):  # toda divisão do conjunto em A → B
            for antecedente in map(frozenset, combinations(conjunto, tamanho)):
                consequente = conjunto - antecedente
                confianca = sup / suporte[antecedente]  # P(B | A)
                regras.append(
                    {
                        "antecedente": " + ".join(sorted(antecedente)),
                        "consequente": " + ".join(sorted(consequente)),
                        "itens": len(conjunto),
                        "suporte": sup,  # P(A e B)
                        "confianca": confianca,
                        "lift": confianca / suporte[consequente],  # P(B | A) / P(B)
                    }
                )
    return pd.DataFrame(regras)


regras = gerar_regras(resultados["FP-Growth"])
fortes = (
    # só regras confiáveis e informativas
    regras.query("confianca >= @MIN_CONFIANCA and lift >= @MIN_LIFT")
    .sort_values(["lift", "confianca"], ascending=False)
    .reset_index(drop=True)
)
print(
    f"{len(regras):,} regras geradas · {len(fortes):,} com confiança ≥ {MIN_CONFIANCA:.0%} "
    f"e lift ≥ {MIN_LIFT}".replace(",", ".")
)

formato = {"suporte": "{:.1%}", "confianca": "{:.1%}", "lift": "{:.2f}"}
fortes.head(TOP).style.format(formato).bar(subset=["lift"], color="#9ecae1")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pontos = ax.scatter(
    regras["suporte"], regras["confianca"], c=regras["lift"], s=8, cmap="viridis", alpha=0.6
)  # cada ponto é uma regra
ax.axhline(MIN_CONFIANCA, color="gray", ls=":")  # corte de confiança
ax.set(title="Regras · suporte × confiança (cor = lift)", xlabel="suporte", ylabel="confiança")
fig.colorbar(pontos, label="lift")
ax.grid(alpha=0.3)
plt.show()

## 4 · O que leva ao cancelamento

As regras mais úteis para a operação são as que têm `is_canceled` como
**consequente**: quais combinações de características indicam uma reserva com
alta ou baixa chance de ser cancelada. Isso é a pergunta 3 do PBL respondida em
cima das mesmas colunas que descrevem o perfil da reserva — sem que o
cancelamento tenha entrado na formação de nenhum grupo.

In [ ]:
for desfecho in ("sim", "nao"):
    # regras que terminam no desfecho
    selecao = fortes[fortes["consequente"] == f"is_canceled={desfecho}"].head(10)
    rotulo = "cancelada" if desfecho == "sim" else "não cancelada"
    display(Markdown(f"**Características → reserva {rotulo}** ({len(selecao)} regras)"))
    display(selecao.style.format(formato).hide(axis="index"))

In [ ]:
regras_saida = RAIZ / "data" / "processed" / "regras_associacao.csv"
fortes.to_csv(regras_saida, sep=";", index=False)  # regras fortes, para consulta
print(f"Salvas {len(fortes)} regras em data/processed/regras_associacao.csv")

## Como ler o que saiu

* **Apriori, Eclat e FP-Growth dão o mesmo resultado**, e a escolha entre eles
  é só de desempenho. Em bases deste tamanho, a diferença é de segundos a
  dezenas de segundos.
* **Regras óbvias aparecem no topo.** `stays_in_week_nights=alto` →
  `total_nights=alto` é verdadeira por construção (`total_nights` é soma de
  `stays_in_weekend_nights` e `stays_in_week_nights`); o mesmo vale para
  `adults=alto` → `total_guests=alto`. São verdadeiras, mas não ensinam nada
  sobre o negócio.
* **Uma armadilha conhecida da base não aparece aqui — e vale registrar por quê.**
  A literatura que acompanha esta base (Antonio, de Almeida & Nunes, 2019)
  documenta que reservas com `deposit_type=Non Refund` cancelam quase sempre
  — o oposto do que a lógica de negócio sugeriria (quem paga sem reembolso
  deveria cancelar menos), um padrão possivelmente ligado a como certas
  agências reportam reservas de grupo, não a uma preferência real do
  hóspede. Nesta base tratada, a confirmação direta é: de 14.587 reservas
  `Non Refund`, 99,4% foram canceladas (contra 28,4% em `No Deposit`). Mas
  `Non Refund` tem suporte de 12,2% — abaixo do corte `MIN_SUPORTE = 15%`
  deste notebook — então **ela nunca chega a virar item frequente e não
  aparece em nenhuma regra listada acima**. É um lembrete de que o corte de
  suporte filtra também padrões fortes e conhecidos, não só ruído; quem
  quiser essa regra específica deve rodar com `MIN_SUPORTE` mais baixo (ex.
  10%) só para esse item, ou investigá-lo fora da mineração de regras.
* **Lift alto com suporte baixo é nicho.** A regra vale para poucas reservas.
  Para decisões de operação (overbooking, canal, antecedência), prefira
  regras com suporte razoável.
* **Associação não é causa.** "Reserva de grupo → cancela" não quer dizer que
  reservar em grupo causa o cancelamento. As duas coisas podem vir juntas por
  um terceiro fator, como o canal de venda.
* **Para ajustar:** se surgirem regras demais, suba o `MIN_SUPORTE` ou o
  `MIN_LIFT`. Se surgirem de menos, desça o `MIN_SUPORTE` (o tempo cresce
  rápido) ou aumente o `MAX_ITENS`.